In [ ]:
import os
os.chdir("/content")
if os.path.exists("structure_properties_fall_2026"):
    !git -C structure_properties_fall_2026 pull origin main
else:
    !git clone https://github.com/aldenbarnes/structure_properties_fall_2026.git
os.chdir("/content/structure_properties_fall_2026")
!ls Code
%pip install -q -r requirements.txt

In [2]:
import os, subprocess, sys
from google.colab import userdata

%pip install -q streamlit
os.chdir("/content/structure_properties_fall_2026")
os.environ["REALTYAPI_KEY"] = userdata.get("REALTYAPI_KEY")

log = open("streamlit.log", "w")
subprocess.Popen([sys.executable, "-m", "streamlit", "run", "Code/tool/rent_lookup_app.py",
                  "--server.headless", "true", "--server.port", "8501"],
                 stdout=log, stderr=log)

<Popen: returncode: None args: ['/usr/bin/python3', '-m', 'streamlit', 'run'...>

In [ ]:
import os, re, subprocess, time

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "cloudflared"], check=True)
    os.chmod("cloudflared", 0o755)
subprocess.run(["pkill", "-f", "cloudflared tunnel"])   # close the tunnel from an earlier run, if any

# The tunnel runs in the background and logs to cloudflared.log; only the link is printed.
log = open("cloudflared.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=log, stderr=subprocess.STDOUT)
link = None
for _ in range(60):
    time.sleep(1)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if found:
        link = found.group(0)
        break
print(link or "No link after 60 seconds; see cloudflared.log")